In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("Day 9 - Fine-tuning BERT")

PyTorch: 2.12.0+cu130
Device: GPU
Day 9 - Fine-tuning BERT


In [2]:
print("=== Loading Dataset ===\n")

# Load IMDb sentiment dataset - 25 k movie reviews, positive/negative labels
# This simulates loading document classification data for your RAG pipeline

dataset = load_dataset("imdb")

print(f"Dataset structure: {dataset}")
print(f"\n Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

# Inspect one sample
sample = dataset['train'][0]
print(f"\nSample review (first 200 chars):")
print(f"  Text: {sample['text'][:200]}...")
print(f"  Lable: {sample['label']} ({'positive' if sample['label'] == 1 else 'negative'})")

# Label distribution
train_labels = dataset['train']['label']
positive = sum(train_labels)
negative = len(train_labels) - positive
print(f"\nLabel distribution:")
print(f"  Positive: {positive} ({positive/len(train_labels)*100:.1f}%)")
print(f"  Negative: {negative} ({negative/len(train_labels)*100:.1f}%)")

=== Loading Dataset ===

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

 Train size: 25000
Test size: 25000

Sample review (first 200 chars):
  Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...
  Lable: 0 (negative)

Label distribution:
  Positive: 12500 (50.0%)
  Negative: 12500 (50.0%)


In [3]:
print("=== Tokenizing Dataset ===\n")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding = "max_length",
        truncation=True,
        max_length=128  # Bert max is 512, using 128 for speed
    )

# Apply tokenization to entire dataset at once
#  This is the huggingFace way  - no manual loops
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,           # Process in batches for speed
    remove_columns=["text"] # remove raw text after tokenizing
)

print(f"Original colums: {dataset['train'].column_names}")
print(f"Tokenized columns: {tokenized_dataset['train'].column_names}")
print(f"\nSample tokenized entry:")
print(f"  input_ids length : {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"  attention_mask length: {len(tokenized_dataset['train'][0]['attention_mask'])}")
print(f"  label: {tokenized_dataset['train'][0]['label']}")

# Use small subset for fast training - 2000 train, 500 test
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_test = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

print(f"\nUsing subset for training:")
print(f"   Train: {len(small_train)} samples")
print(f"   Test:  {len(small_test)} samples")

=== Tokenizing Dataset ===

Original colums: ['text', 'label']
Tokenized columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']

Sample tokenized entry:
  input_ids length : 128
  attention_mask length: 128
  label: 0

Using subset for training:
   Train: 2000 samples
   Test:  500 samples


In [4]:
print("=== Loading BERT for Classification ===\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained BERT with a classification on top
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels =2 # positve/negative
)
model = model.to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Model on: {device}")

# Evaluation metric
import evaluate
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# Training arguments
training_args = TrainingArguments(
    output_dir="./bert-sentiment",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps= 100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# Trainer 
trainer =Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

print(f"\nTraining config:")
print(f"   Epochs:    {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_eval_batch_size}")
print(f"  Steps:      {len(small_train)  // training_args.per_device_train_batch_size*training_args.num_train_epochs}")
print(f"\nReady to Train")

=== Loading BERT for Classification ===



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 109,483,778
Trainable parameters: 109,483,778
Model on: cuda

Training config:
   Epochs:    2
  Batch size: 32
  Steps:      250

Ready to Train


In [5]:
print("=== Training BERT ===\n")
print("This will take 5-10 minutes on rtx 3050...\n")

# Train
trainer.train()

print(f"\n=== Training Complete ===")

=== Training BERT ===

This will take 5-10 minutes on rtx 3050...



Epoch,Training Loss,Validation Loss,Accuracy
1,0.494800,0.496174,0.814000
2,0.265300,0.368031,0.858000



=== Training Complete ===


In [6]:
print("=== Evaluation ===\n")

# Final evaluation
results = trainer.evaluate()
print(f"Final Accuracy: {results['eval_accuracy']:.4f}")
print(f"Final Loss:     {results['eval_loss']:.4f}")

# Test on custom sentences
print("\n=== Testing on RAG-specific sentences ===\n")

from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_sentences = [
    "RAG pipeline retrieves highly relevant document",
    "The system keeps returning completely wrong answers",
    "Hybrid search improved out retrieval accuracy significantly",
    "The model hallucinates facts that are not in the documents",
    "Fine tuning on domain data made the model much better",
    "The pipeline is too slow and crashes under heavy load"
]

label_map = {
    "LABEL_0": "NEGATIVE",
    "LABEL_1": "POSITIVE"
}

for sentence in test_sentences:
    result = classifier(sentence, truncation=True, max_length=128)[0]
    label = label_map[result['label']]
    score = result['score']
    emoji = "✅" if label == "POSITIVE" else "❌"
    print(f"{emoji}[{label} {score:.4f}] {sentence}")

=== Evaluation ===



Device set to use cuda:0


Final Accuracy: 0.8580
Final Loss:     0.3680

=== Testing on RAG-specific sentences ===

✅[POSITIVE 0.8186] RAG pipeline retrieves highly relevant document
❌[NEGATIVE 0.9501] The system keeps returning completely wrong answers
✅[POSITIVE 0.6538] Hybrid search improved out retrieval accuracy significantly
❌[NEGATIVE 0.7989] The model hallucinates facts that are not in the documents
✅[POSITIVE 0.8026] Fine tuning on domain data made the model much better
❌[NEGATIVE 0.9369] The pipeline is too slow and crashes under heavy load


In [7]:
HF_USERNAME = "Havoc-Jay"

# Save locally first
trainer.save_model("./bert-sentiment-final")
tokenizer.save_pretrained("./bert-sentiment-final")

# Push using model directly
model.push_to_hub(f"{HF_USERNAME}/bert-sentiment-rag")
tokenizer.push_to_hub(f"{HF_USERNAME}/bert-sentiment-rag")

print(f"\nModel pushed successfully!")
print(f"View at: https://huggingface.co/{HF_USERNAME}/bert-sentiment-rag")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]


Model pushed successfully!
View at: https://huggingface.co/Havoc-Jay/bert-sentiment-rag
